# **Méteo et Pollution - Projet Pyhton**

In [ ]:
# Liste des imports
import logging
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
from scipy.interpolate import PchipInterpolator

# Dans le projet
from src import fct
from src.extract import trafic
from src.extract.trafic import RACINE, lire_config
from src import stockage
from src.clean import trafic as nettoyage_trafic

# les fonctions de nettoyage journalisent le nombre de lignes concernées : on les affiche
logging.basicConfig(level=logging.INFO, format="%(message)s")

## Téléchargement des datasets

In [ ]:
RELANCER_EXTRACTION_TRAFIC = False

Les datasets `airparif` et `trafic` sont très volumineux et prennent ≈1h chacun à être téléchargés depuis le début. Pour gagner du temps, les versions brutes sont enregistrées dans des buckets SSP Cloud personnels et sont récupérés là-bas directement.

### **Dataset Meteo**

In [ ]:
import importlib
from src import fct

importlib.reload(fct)

df_raw_meteo = fct.meteo_dataset_extraction()

print("Dimensions :", df_raw_meteo.shape)
display(df_raw_meteo.head())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df_meteo = df_raw_meteo.copy()

#premier tris des variables qui ne vont pas être très utile au projet

df_meteo = df_meteo.drop(columns=[
    "apparent_temperature",
    "dew_point_2m",
    "snow_depth",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
    "soil_temperature_0_to_7cm",
    "soil_moisture_0_to_7cm",
])

display(df_meteo.isna().sum())
display(df_meteo.dtypes)

#Transformation du format des dates, pour un meilleur gestions de celles-ci avec pandas
df_meteo["time"] = pd.to_datetime(df_meteo["time"], utc=True)
df_meteo = df_meteo.set_index("time", drop=False).sort_index()

temperature_jour = df_meteo["temperature_2m"].resample("D").mean()
temperature_mois = df_meteo["temperature_2m"].resample("MS").mean()

#### Température moyenne par jour

Chaque point de la courbe représente la moyenne des températures horaires d'une journée.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
ax.plot(temperature_jour.index, temperature_jour, color="#d97706", linewidth=1.5)

ax.set_title("Température moyenne journalière à Paris")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Température moyenne (°C)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
ax.margins(x=0.01)
plt.show()

#### Température moyenne par mois

Chaque barre représente la moyenne de toutes les températures horaires du mois.
Les mois sont présentés dans l'ordre chronologique, de septembre 2025 à août 2026.

In [ ]:
mois_fr = [
    "Janv.", "Févr.", "Mars", "Avr.", "Mai", "Juin",
    "Juil.", "Août", "Sept.", "Oct.", "Nov.", "Déc.",
]
labels_mois = [
    f"{mois_fr[date.month - 1]}\n{date.year}" for date in temperature_mois.index
]

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
barres = ax.bar(labels_mois, temperature_mois.to_numpy(), color="#2563eb", width=0.65)
ax.bar_label(barres, fmt="%.1f °C", padding=4, fontsize=9)

ax.set_title("Température moyenne mensuelle à Paris")
ax.set_xlabel("Mois (UTC)")
ax.set_ylabel("Température moyenne (°C)")
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.25)
ax.margins(y=0.15)
plt.show()

#### Pression en surface et couverture nuageuse

Chaque point correspond aux deux mesures prises à la même heure. La transparence
permet de repérer les zones où les observations se superposent.

In [ ]:
# Écarte seulement les heures où une des deux mesures est manquante.
pression_nuages = df_meteo[["surface_pressure", "cloud_cover"]].dropna()

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.scatter(
    pression_nuages["surface_pressure"],
    pression_nuages["cloud_cover"],
    s=12,
    alpha=0.2,
    edgecolors="none",
)

ax.set_title("Pression en surface et couverture nuageuse à Paris")
ax.set_xlabel("Pression en surface (hPa)")
ax.set_ylabel("Couverture nuageuse (%)")
ax.set_ylim(-3, 103)
ax.grid(alpha=0.2)
ax.set_axisbelow(True)
plt.show()

#### Fréquence de la pluie selon la pression atmosphérique

On utilise la pression **au niveau de la mer** (`pressure_msl`), comme sur le
graphique de référence. Une heure est considérée comme pluvieuse si `rain > 0 mm`.
Pour chaque tranche de **5 hPa**, on calcule :

**Fréquence (%) = 100 × nombre d'heures pluvieuses / nombre total d'heures.**

Les points sont placés au centre des tranches. La courbe elle sert de guide
visuel, pour montrer le lien entre pluie et pression. C'est d'ailleurs comme cela que la meteo est prédite par les baromètres mécanique accrochés chez nos grands parents. A noter que les pressions les plus basses ont ici peu d'observations.

In [ ]:


# On créée une nouvelle data frame pandans qui va regrouper les donners dans ces tranches de pressions
df_pluie_pression = df_meteo[["pressure_msl", "rain"]].dropna().copy()
df_pluie_pression["heure_pluvieuse"] = df_pluie_pression["rain"] > 0

# Tranches de 5 hPa couvrant toutes les pressions observées.
pas_pression = 5
pression_min = pas_pression * np.floor(df_pluie_pression["pressure_msl"].min() / pas_pression)
pression_max = pas_pression * np.floor(df_pluie_pression["pressure_msl"].max() / pas_pression)
bornes_pression = np.arange(pression_min, pression_max + 2 * pas_pression, pas_pression) #le +2 * pas pression est ici pour ne pas manquer les valeurs qu'on aurait pu oublier à cause de l'arrondit au multiple de 5 avec la fonction np.floor
df_pluie_pression["tranche_pression"] = pd.cut(
    df_pluie_pression["pressure_msl"], bins=bornes_pression, right=False
) #ici, cut va permettre d'attribuer une tranche de pression à chaque heure d'observation de pression

stats_pression_pluie = df_pluie_pression.groupby("tranche_pression").agg(
    nb_heures=("heure_pluvieuse", "size"), #nombre d'heure total observée dans la tranche
    nb_heures_pluvieuses=("heure_pluvieuse", "sum"), # nombre d'heure total dee pluie dans la tranche de pression
    frequence_pluie=("heure_pluvieuse", "mean"), # pourcentage d'heures pluviuse dans la tranche de pression
) # on regroupe les heures appartennant à une même tranche, en leur créeant des statistiques qui vont nous être utile pour créer le tableau final
stats_pression_pluie["frequence_pluie"] *= 100
stats_pression_pluie["pression_centrale"] = [
    tranche.mid for tranche in stats_pression_pluie.index
] 

pressions = stats_pression_pluie["pression_centrale"].to_numpy()
frequences = stats_pression_pluie["frequence_pluie"].to_numpy()

fig, ax = plt.subplots(figsize=(14, 7), constrained_layout=True)
if len(pressions) >= 2:
    pression_lisse = np.linspace(pressions.min(), pressions.max(), 500)
    interpolation = PchipInterpolator(pressions, frequences, extrapolate=False)
    ax.plot(
        pression_lisse, interpolation(pression_lisse),
        color="royalblue", linewidth=2.5, label="Courbe interpolée",
    ) # ici, la méthode permet d'interpoler les points créés, afin de créer une jolie courbe qui va les relier. La courbe n'a en elle même pas une vraie signification scientifique
ax.scatter(
    pressions, frequences, color="navy", s=48, zorder=3,
    label="Fréquences observées",
)

ax.set_title(
    "Fréquence de la pluie selon la pression atmosphérique à Paris",
    fontsize=15, fontweight="bold",
)
ax.set_xlabel("Pression atmosphérique au niveau de la mer (hPa)", fontsize=12)
ax.set_ylabel("Fréquence des heures pluvieuses (%)", fontsize=12)
ax.set_ylim(0, 100)
ax.set_xlim(bornes_pression[0] - 1, bornes_pression[-1] + 1)
ax.set_yticks(np.arange(0, 101, 20))
ax.grid(alpha=0.2)
ax.set_axisbelow(True)
ax.legend(loc="upper right")
plt.show()

### **Dataset trafic**

In [ ]:
CFG = lire_config()

if RELANCER_EXTRACTION_TRAFIC:
    trafic.main()
elif not (RACINE / CFG["chemins"]["raw"] / "trafic").exists():
    stockage.recuperer(CFG, dataset="trafic")

#### Nettoyage

Le nettoyage utilise les fonctions de `src/clean/trafic.py`, dans le même ordre que `uv run python -m src.clean.trafic`. Le détail de chaque étape, avec les graphiques qui justifient les choix, est dans `notebooks/analyse_trafic.ipynb`. Les seuils sont dans `config.yaml`.

1. On garde `iu_ac` (identifiant de l'arc), `q` (débit, véh/h) et `k` (taux d'occupation, %). `t_1h` marque la **fin** de l'heure : on le décale d'une heure pour obtenir `heure`, le début de période en UTC.
2. Le référentiel des arcs contient d'anciennes versions de certains arcs : on garde la plus récente.
3. On retire les arcs sans géométrie, qu'on ne pourrait pas rattacher à une station.
4. Plus de 24 h de suite à zéro : le capteur est en panne, ces valeurs passent à `NaN`.
5. On garde les arcs qui ont un débit pour au moins 70 % des heures.
6. On complète la grille : une ligne par arc et par heure, avec `NaN` pour les heures absentes.
7. On ajoute le libellé, la longueur et le centroïde (Lambert-93) de chaque arc.

In [ ]:
SEUIL_ZEROS = CFG["trafic"]["zeros_consecutifs_max"]
SEUIL_DEBIT = CFG["trafic"]["part_sans_debit_max"]


def resume_trafic(df, etape):
    """Lignes, arcs et valeurs renseignées après une étape du nettoyage."""
    return {"étape": etape, "lignes": len(df), "arcs": df["iu_ac"].nunique(),
            "débits renseignés": df["q"].notna().sum(), "taux renseignés": df["k"].notna().sum()}


df_raw_trafic, ref_trafic = nettoyage_trafic.charger_brut(RACINE / CFG["chemins"]["raw"] / "trafic")
df_trafic = nettoyage_trafic.preparer(df_raw_trafic)
etapes_trafic = [resume_trafic(df_trafic, "données brutes")]

arcs = nettoyage_trafic.dedoublonner_referentiel(ref_trafic)
df_trafic = nettoyage_trafic.retirer_arcs_sans_geo(df_trafic, arcs)
etapes_trafic.append(resume_trafic(df_trafic, "arcs sans géométrie"))

for colonne in ["q", "k"]:
    df_trafic = nettoyage_trafic.retirer_longues_series_de_zeros(df_trafic, colonne, SEUIL_ZEROS)
etapes_trafic.append(resume_trafic(df_trafic, f"séries de zéros de plus de {SEUIL_ZEROS} h"))

#### Choix des arcs gardés

Beaucoup d'arcs n'ont presque jamais de débit, d'autres presque toujours : la distribution est bimodale. On garde ceux qui ont un débit pour au moins 70 % des heures.

In [ ]:
part_sans_debit = nettoyage_trafic.part_heures_sans_debit(df_trafic).sort_values()
n_gardes = (part_sans_debit <= SEUIL_DEBIT).sum()

# part des 24 h de chaque jour avec un débit, par arc (arcs triés comme part_sans_debit)
jour = df_trafic["heure"].dt.floor("D")
jours = pd.date_range(jour.min(), jour.max(), freq="D")
dispo = df_trafic["q"].notna().groupby([df_trafic["iu_ac"], jour]).sum().unstack(fill_value=0)
dispo = dispo.reindex(index=part_sans_debit.index, columns=jours, fill_value=0) / 24
debuts_mois = np.where(jours.day == 1)[0]

fig, (ax_hist, ax_jours) = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
fig.suptitle(f"Les arcs ont un débit presque toujours ou presque jamais : {n_gardes} sur {len(part_sans_debit)} sont gardés")

ax_hist.hist(part_sans_debit * 100, bins=50, color="#2563eb")
ax_hist.axvline(SEUIL_DEBIT * 100, color="#dc2626", linewidth=1.5, label=f"seuil retenu : {SEUIL_DEBIT:.0%}")
ax_hist.set_title("Distribution par arc")
ax_hist.set_xlabel("Part des heures sans débit sur la période (%)")
ax_hist.set_ylabel("Nombre d'arcs")
ax_hist.grid(axis="y", alpha=0.25)
ax_hist.legend()

image = ax_jours.imshow(dispo.to_numpy(), aspect="auto", cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
ax_jours.axhline(n_gardes - 0.5, color="#dc2626", linewidth=1.5, label=f"seuil : {n_gardes} arcs gardés au-dessus")
ax_jours.set_title("Détail par arc et par jour")
ax_jours.set_xticks(debuts_mois, jours[debuts_mois].strftime("%m/%Y"), rotation=45)
ax_jours.set_xlabel("Date (UTC)")
ax_jours.set_ylabel("Arcs, triés par part d'heures sans débit")
ax_jours.legend(loc="lower left")
fig.colorbar(image, ax=ax_jours, shrink=0.8, label="Part des 24 h du jour avec un débit")
plt.show()

df_trafic = nettoyage_trafic.garder_arcs_mesures(df_trafic, SEUIL_DEBIT)
etapes_trafic.append(resume_trafic(df_trafic, f"arcs sans débit plus de {SEUIL_DEBIT:.0%} du temps"))

In [ ]:
df_trafic = nettoyage_trafic.completer_grille(df_trafic)
etapes_trafic.append(resume_trafic(df_trafic, "grille complète"))
df_trafic = nettoyage_trafic.ajouter_infos_arcs(df_trafic, arcs)
del df_raw_trafic

assert len(df_trafic) == df_trafic["iu_ac"].nunique() * df_trafic["heure"].nunique()
assert not df_trafic.duplicated(["iu_ac", "heure"]).any()

# même sortie que `uv run python -m src.clean.trafic`
nettoyage_trafic.enregistrer(df_trafic, arcs, RACINE / CFG["chemins"]["interim"] / "trafic")

display(pd.DataFrame(etapes_trafic).set_index("étape"))
print(f"Débit manquant : {df_trafic['q'].isna().mean():.1%} | "
      f"taux d'occupation manquant : {df_trafic['k'].isna().mean():.1%}")
display(df_trafic.head())

On passe de 23 489 736 lignes et 2 980 arcs à 12 272 496 lignes : 1 531 arcs × 8 016 heures, du 31/10/2025 23:00 au 30/09/2026 22:00 UTC. Le seuil de 70 % retire presque la moitié des arcs mais seulement 11 % des débits renseignés : ce sont surtout des arcs vides. Il manque encore 8,6 % des débits et 23,2 % des taux d'occupation (150 arcs gardés n'ont jamais de `k`) ; rien n'est imputé à ce stade.

Le fichier `data/interim/trafic/trafic.parquet` ne contient pas le tracé des arcs, seulement leur centroïde : le tracé est dans `data/interim/trafic/referentiel_nettoye.parquet`, à joindre par `iu_ac`.

### Exploration des données de trafic nettoyées

#### Distribution du débit

In [ ]:
display(df_trafic["q"].describe().to_frame("débit (véh/h)").T.round(0))

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
ax.hist(df_trafic["q"].dropna(), bins=80, color="#2563eb")
ax.set_yscale("log")
ax.axvline(df_trafic["q"].median(), color="#dc2626", linewidth=1.5,
           label=f"médiane : {df_trafic['q'].median():.0f} véh/h")
ax.set_title("La plupart des débits sont sous 2 000 véh/h, avec un second groupe vers 5 000 véh/h (périphérique)")
ax.set_xlabel("Débit (véh/h)")
ax.set_ylabel("Nombre d'heures × arcs (échelle log)")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()

#### Rythme de la semaine

Débit moyen d'un arc pour chaque heure de la semaine (moyenne sur toute la période).

In [ ]:
jours_semaine = ["Lun.", "Mar.", "Mer.", "Jeu.", "Ven.", "Sam.", "Dim."]
heure_semaine = df_trafic["heure"].dt.dayofweek * 24 + df_trafic["heure"].dt.hour  # 0 = lundi 0 h
profil_semaine = df_trafic.groupby(heure_semaine)["q"].mean()

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
ax.plot(profil_semaine.index, profil_semaine.to_numpy(), color="#2563eb", linewidth=1.8)
for jour in range(1, 7):
    ax.axvline(24 * jour, color="grey", linewidth=0.8, alpha=0.5)
ax.set_xticks(range(12, 168, 24), jours_semaine)
ax.set_xlim(0, 167)
ax.set_title("Le trafic suit le rythme des journées : creux la nuit, plus faible le week-end")
ax.set_xlabel("Heure de la semaine (UTC)")
ax.set_ylabel("Débit moyen par arc (véh/h)")
ax.grid(axis="y", alpha=0.25)
plt.show()

#### Carte du débit moyen

Chaque arc gardé est coloré selon son débit moyen sur la période ; les autres arcs du référentiel sont en gris clair.

In [ ]:
arcs_gardes = arcs[arcs["iu_ac"].isin(df_trafic["iu_ac"])]
q_moyen = df_trafic.groupby("iu_ac")["q"].mean().rename("q_moyen")
carte = arcs_gardes.merge(q_moyen, left_on="iu_ac", right_index=True).sort_values("q_moyen")  # forts débits dessinés en dernier

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
arcs.plot(ax=ax, color="#e5e7eb", linewidth=0.5)
carte.plot(ax=ax, column="q_moyen", cmap="viridis", norm=LogNorm(), linewidth=1.5, legend=True,
           legend_kwds={"label": "Débit moyen (véh/h, échelle log)", "shrink": 0.6})
ax.set_title("Le périphérique et les grands axes concentrent le trafic")
ax.set_axis_off()
plt.show()

#### Valeurs manquantes

Part des débits manquants chaque jour, sur les arcs gardés.

In [ ]:
manquant_jour = df_trafic["q"].isna().groupby(df_trafic["heure"].dt.floor("D")).mean() * 100

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
ax.plot(manquant_jour.index, manquant_jour.to_numpy(), color="#2563eb", linewidth=1.5)
ax.axhline(manquant_jour.median(), color="grey", linestyle="--", linewidth=1,
           label=f"médiane : {manquant_jour.median():.1f} %")
ax.set_ylim(0, 100)
ax.set_title("Peu de débits manquent au quotidien, sauf pendant cinq coupures de la collecte")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Débits manquants (%)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.show()

print("Jours avec plus de 50 % de débits manquants :",
      ", ".join(manquant_jour[manquant_jour > 50].index.strftime("%d/%m/%Y")))

#### Ce qu'on retient pour la suite

- Le débit varie beaucoup d'un arc à l'autre (médiane 297 véh/h, jusqu'à près de 12 000 sur le périphérique) : à la fusion, il faudra agréger les arcs autour de chaque station (débit moyen ou total, nombre d'arcs actifs) plutôt que de les comparer un à un.
- Le trafic a un rythme journalier et hebdomadaire très net, qu'on s'attend à retrouver dans le NO₂ ; les variables de calendrier (heure, jour, week-end) risquent donc d'être en partie redondantes avec le trafic.
- Cinq journées (29/12, 29/01, 16/02, 24/03, 13/04) n'ont presque aucune mesure : elles devront être traitées à l'imputation, ou signalées par un indicateur de manque.

### Exploration des données météo à Paris

Les températures correspondent aux mesures à 2 mètres du sol. Les moyennes
journalières et mensuelles sont calculées à partir des observations horaires.
Les jours et les mois sont définis en **UTC**, comme dans l'extraction.